# 第 43 课：RNN-T 与 TDT——二维对齐和跳帧

这一课从你已掌握的 CTC 出发，回答三个问题：RNN-T 为什么比 CTC 多一个维度？它怎样在不看未来的情况下利用已输出文字？TDT 为什么可以减少大量 blank 解码步骤？

前置：第 10～14 课与第 42 课。CPU 约需 1～2 分钟。

## 完成标准

完成后你应能：

1. 画出 RNN-T 的 encoder、predictor、joiner；
2. 解释 logits `[B,T,U+1,V]` 的每一维；
3. 在二维 lattice 上区分 blank 边和 label 边；
4. 从空白写出 log-space forward recurrence；
5. 解释 TDT 的 token head 与 duration head 怎样减少解码步数。

本课实现教学版单样本 RNN-T loss，用暴力枚举验证数值与梯度。生产训练应使用经过优化和测试的 RNNT loss kernel。

## 课前诊断（先回答）

1. CTC 的每一帧预测是否依赖此前已经输出的 token？
2. 若音频有 `T=100` 个编码帧、转录有 `U=12` 个 token，RNN-T lattice 有哪两个坐标？
3. 流式模型为什么不能依赖完整句子的未来帧？

In [ ]:
import itertools
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(13)
np.random.seed(13)
random.seed(13)
torch.set_num_threads(2)
print("torch:", torch.__version__)

## 1. CTC 与 RNN-T 的根本差别

CTC 在时间步 $t$ 只根据声学编码器表示预测：

$$P(k\mid h_t)$$

RNN-T 另外使用 predictor 表示此前已经输出的 token 历史：

$$P(k\mid h_t, g_u)$$

```text
音频特征 → Encoder → h[t] ─┐
                             ├→ Joiner → token/blank
历史 token → Predictor → g[u]┘
```

- `t`：已经处理到哪个声学时间步；
- `u`：已经输出了多少个目标 token；
- blank：时间前进一步 `(t,u) → (t+1,u)`；
- 下一个正确 label：输出 token，但时间不前进 `(t,u) → (t,u+1)`。

所以 joint logits 是 `[B,T,U+1,V]`，而不是 CTC 的 `[B,T,V]`。

In [ ]:
class TinyRNNT(nn.Module):
    def __init__(self, feat_dim=12, vocab_size=6, hidden=16, blank=0):
        super().__init__()
        self.blank = blank
        self.vocab_size = vocab_size
        self.encoder = nn.GRU(feat_dim, hidden, batch_first=True)
        self.embedding = nn.Embedding(vocab_size, hidden)
        self.predictor = nn.GRU(hidden, hidden, batch_first=True)
        self.enc_proj = nn.Linear(hidden, hidden)
        self.pred_proj = nn.Linear(hidden, hidden)
        self.output = nn.Linear(hidden, vocab_size)

    def encode(self, features):
        return self.encoder(features)[0]

    def predict(self, targets):
        # 在目标前加 blank 作为教学版 BOS，得到 U+1 个 predictor 状态。
        bos = torch.full(
            (targets.size(0), 1), self.blank,
            dtype=targets.dtype, device=targets.device,
        )
        tokens_in = torch.cat([bos, targets], dim=1)
        embedded = self.embedding(tokens_in)
        return self.predictor(embedded)[0]

    def join(self, enc, pred):
        # [B,T,1,H] + [B,1,U+1,H] → [B,T,U+1,V]
        joint = torch.tanh(
            self.enc_proj(enc).unsqueeze(2)
            + self.pred_proj(pred).unsqueeze(1)
        )
        return self.output(joint)

    def forward(self, features, targets):
        return self.join(self.encode(features), self.predict(targets))


model = TinyRNNT()
features = torch.randn(2, 7, 12)
targets = torch.tensor([[1, 2, 3], [3, 2, 1]])
logits = model(features, targets)
print("features:", tuple(features.shape))
print("targets:", tuple(targets.shape))
print("joint logits:", tuple(logits.shape))
assert logits.shape == (2, 7, 4, 6)
print("断言通过：U 个目标需要 U+1 个 predictor 状态。")

## 2. 二维 forward 动态规划

令 $\alpha(t,u)$ 表示到达 lattice 状态 `(t,u)` 的所有路径概率之和（在 log-space 中）。两种转移为：

$$
\alpha(t+1,u)\;\mathrel{\oplus}=\;\alpha(t,u)+\log p(\text{blank}\mid t,u)
$$

$$
\alpha(t,u+1)\;\mathrel{\oplus}=\;\alpha(t,u)+\log p(y_u\mid t,u)
$$

$\oplus$ 是 `logaddexp`，对应普通概率空间的加法。最终 loss 为 $-\alpha(T,U)$。

In [ ]:
def rnnt_loss_single(log_probs, targets, blank=0):
    """教学版 RNN-T forward loss。

    log_probs: [T,U+1,V]
    targets: [U]
    """
    T, U1, _ = log_probs.shape
    U = targets.numel()
    assert U1 == U + 1

    neg_inf = log_probs.new_tensor(float("-inf"))
    alpha = [[neg_inf for _ in range(U + 1)] for _ in range(T + 1)]
    alpha[0][0] = log_probs.new_zeros(())

    for t in range(T + 1):
        for u in range(U + 1):
            current = alpha[t][u]
            if t < T:  # blank 消耗一个声学帧
                candidate = current + log_probs[t, u, blank]
                alpha[t + 1][u] = torch.logaddexp(alpha[t + 1][u], candidate)
            if t < T and u < U:  # label 消耗一个目标 token
                label = int(targets[u])
                candidate = current + log_probs[t, u, label]
                alpha[t][u + 1] = torch.logaddexp(alpha[t][u + 1], candidate)

    return -alpha[T][U], alpha


tiny_logits = torch.randn(2, 3, 4, requires_grad=True)  # T=2,U=2,V=4
tiny_targets = torch.tensor([1, 2])
tiny_log_probs = tiny_logits.log_softmax(-1)
loss, alpha = rnnt_loss_single(tiny_log_probs, tiny_targets)
loss.backward()

print(f"loss={loss.item():.6f}")
print("alpha(T,U)=", alpha[2][2].item())
print("gradient norm=", tiny_logits.grad.norm().item())
assert math.isfinite(loss.item()) and tiny_logits.grad.norm() > 0
print("断言通过：forward loss 有限且能反向传播。")

### 为什么必须把所有合法路径相加？

同一个转录可能对应许多对齐。例如目标 `[A,B]`、`T=2` 时，可以先输出 A，也可以先 blank 再输出 A。训练数据没有逐帧对齐标注，RNN-T loss 必须边缘化所有合法路径。

下面用递归暴力枚举这个极小例子，并与动态规划比较。真实输入的路径数会爆炸，因此生产训练绝不能暴力枚举。

In [ ]:
def enumerate_path_logps(log_probs, targets, t=0, u=0, score=None):
    T, _, _ = log_probs.shape
    U = targets.numel()
    score = log_probs.new_zeros(()) if score is None else score
    if t == T and u == U:
        return [score]
    paths = []
    if t < T:
        paths += enumerate_path_logps(
            log_probs, targets, t + 1, u,
            score + log_probs[t, u, 0],
        )
    if t < T and u < U:
        paths += enumerate_path_logps(
            log_probs, targets, t, u + 1,
            score + log_probs[t, u, int(targets[u])],
        )
    return paths


with torch.no_grad():
    path_scores = enumerate_path_logps(tiny_log_probs.detach(), tiny_targets)
    brute_log_prob = torch.logsumexp(torch.stack(path_scores), dim=0)
    dp_log_prob = -rnnt_loss_single(tiny_log_probs.detach(), tiny_targets)[0]

print("合法路径数:", len(path_scores))
print(f"暴力枚举 log P={brute_log_prob.item():.7f}")
print(f"动态规划 log P={dp_log_prob.item():.7f}")
assert torch.allclose(brute_log_prob, dp_log_prob, atol=1e-6)
print("断言通过：动态规划与所有合法路径之和完全一致。")

## 3. Predictor 为什么让 RNN-T 比 CTC 更像“声学 + 语言”联合模型？

同一声学帧 `h[t]` 会和不同历史状态 `g[u]` 组合。例如已经输出“北”时，joiner 对“京”的评分可以不同于已经输出“背”时的评分。

但 predictor 不是一个可以随意替代的大语言模型：它通常规模较小，且只在 ASR 配对数据上学习。领域热词、专名和长上下文仍需专门设计与评估。

In [ ]:
model.eval()
enc = model.encode(torch.randn(1, 5, 12))
history_a = torch.tensor([[1, 2]])
history_b = torch.tensor([[4, 5]])
pred_a = model.predict(history_a)
pred_b = model.predict(history_b)

with torch.no_grad():
    dist_a = model.join(enc[:, 2:3], pred_a[:, -1:]).softmax(-1).squeeze()
    dist_b = model.join(enc[:, 2:3], pred_b[:, -1:]).softmax(-1).squeeze()

print("相同声学帧，不同 token 历史的最大概率差:", (dist_a-dist_b).abs().max().item())
assert not torch.allclose(dist_a, dist_b)
print("断言通过：predictor 历史会改变 joiner 的输出分布。")

## 4. TDT：同时预测 token 与 duration

标准 Transducer 解码中，blank 通常只让时间前进一帧。长音频会产生大量 blank 步骤。Token-and-Duration Transducer（TDT）增加 duration 分布：

$$P(v,d\mid t,u)=P_T(v\mid t,u)P_D(d\mid t,u)$$

- token head 决定输出 blank 或文字 token；
- duration head 决定时间向前跳多少帧；
- 预测较大 duration 时可以一次跳过多帧，减少 joiner 调用。

下面不是完整 TDT loss，而是一个透明的解码复杂度实验。

In [ ]:
def transducer_blank_steps(num_frames):
    # 最简化的标准 transducer：至少逐帧前进。
    return list(range(num_frames))


def tdt_duration_steps(num_frames, durations):
    visited = []
    t = 0
    i = 0
    while t < num_frames:
        visited.append(t)
        duration = max(1, int(durations[i % len(durations)]))
        t += duration
        i += 1
    return visited


T = 30
standard = transducer_blank_steps(T)
tdt = tdt_duration_steps(T, durations=[4, 3, 5])
print("标准逐帧访问次数:", len(standard), standard)
print("TDT 跳帧访问次数:", len(tdt), tdt)
print(f"本例 joiner 步数减少 {(1-len(tdt)/len(standard))*100:.1f}%")
assert tdt[0] == 0 and len(tdt) < len(standard)
print("断言通过：duration 预测可以跳过中间声学帧。")

### TDT 的边界

跳帧不是免费午餐：duration 预测错了可能跨过重要声学证据。训练需要正确归一化所有 token-duration 对齐，解码也要限制合法 duration。生产实现应使用 NeMo 等经过验证的 loss 和解码器，而不是本课的复杂度演示。

## 5. CTC、RNN-T、TDT 选型

| 目标 | 更自然的起点 | 原因 |
|---|---|---|
| 最简单训练与批量离线推理 | CTC | `[B,T,V]`，高度并行 |
| 实时、利用输出历史 | RNN-T | 原生流式，predictor 建模历史 |
| 高吞吐 Transducer | TDT | duration head 减少 blank/逐帧步骤 |

最终选择必须在自己的数据上比较 CER/WER、RTF、首字延迟、尾延迟、增量稳定性和内存，不能只引用论文中的单一数字。

## 分层练习（24 分）

### A. 回忆（每题 1 分）

1. RNN-T 的三个网络分别叫什么？
2. blank 边改变 `t` 还是 `u`？label 边呢？
3. 为什么 predictor 输入需要 BOS？
4. TDT 比 RNN-T 多预测什么？

### B. 推理（每题 2 分）

5. `T=80,U=10,V=5000` 时 joint logits 的 shape 是什么？
6. 为什么不能直接保存大 batch 的完整 `[B,T,U+1,V]`？
7. 目标有相邻重复 token 时，RNN-T 是否像 CTC 一样必须插 blank？解释。
8. duration 总预测偏大时，可能出现什么识别错误？

### C. 编程与排错（每题 3 分）

9. 把暴力枚举改为同时返回 B/L 路径字符串。
10. 用 `T=3,U=2` 再验证动态规划，观察路径数。
11. 故意把 label 边写成 `(t+1,u+1)`，说明它变成了什么限制。
12. 从空白重写 `rnnt_loss_single`，通过枚举一致性断言。

达到 19/24 且能解释 lattice，才进入自监督预训练。

## 离场小测（闭卷发给老师）

1. 用一句话说出 CTC 与 RNN-T 条件概率的差别。
2. 画一个 `T=3,U=2` 的 lattice，标出 blank 和 label 边。
3. 为什么 RNN-T 能流式，而普通双向 AED 通常不能？
4. TDT 的速度来自哪里？它会引入什么风险？

请附上本课所有断言是否通过，以及你最没有把握的一题。